# 🎯 Clase 1 · Sistemas de Recomendación y Aprendizaje por Refuerzo

**Módulo:** Científico de Datos e Inteligencia Artificial Aplicada · **Unidad 3:** IA Aplicada, Databricks y Arquitecturas Modernas

---

### 🏢 El caso: la plataforma de cursos de DevSenior Code
La academia tiene **40 cursos** y **600 estudiantes** que los califican de 1 a 5. Hoy la página de inicio muestra a todos los mismos cursos.
El equipo de producto nos pide dos cosas:

| Pedido | Pregunta de negocio | Técnica |
|---|---|---|
| **1. "Te puede interesar…"** | ¿Qué curso le mostramos a *cada* estudiante? | **Sistemas de recomendación** (Parte A) |
| **2. Ruta de aprendizaje** | ¿Qué banner destacar y en qué **orden** tomar los cursos para llegar al proyecto final sin abandonar? | **Aprendizaje por refuerzo** (Parte B) |

### 🎯 Objetivos
1. Explicar los tipos de recomendadores: popularidad, contenido, colaborativo e híbrido.
2. Construir recomendadores con **Scikit-Learn** (TF-IDF, similitud coseno, SVD).
3. Entrenar una **factorización de matrices** y un **Neural Collaborative Filtering** con **TensorFlow**.
4. Evaluar con RMSE, precision@k, recall@k y cobertura; resolver el arranque en frío.
5. Entender agente, entorno, estado, acción y recompensa; equilibrar **exploración vs explotación** con bandits.
6. Diseñar una estrategia de aprendizaje secuencial con **Q-learning** (y una **DQN** opcional).

## 📚 Teoría: ¿qué es un sistema de recomendación?
Un recomendador estima **cuánto le va a gustar un ítem a un usuario** que aún no lo ha visto, y muestra los mejores. Netflix, Spotify, Amazon, YouTube, LinkedIn y Rappi viven de esto: en Netflix más del 80 % de lo que se ve viene de una recomendación.

| Enfoque | Idea | Necesita | Fortaleza | Debilidad |
|---|---|---|---|---|
| **Popularidad** | "Lo más visto/mejor calificado" | Solo conteos | Simple, sin arranque en frío | Igual para todos |
| **Basado en contenido** | "Te gustó un curso de Python → otro de Python" | Descripción de los ítems | Funciona con ítems nuevos | No sorprende (burbuja) |
| **Colaborativo** | "Estudiantes como tú también tomaron…" | Historial de muchos usuarios | Descubre gustos no obvios | Arranque en frío, dispersión |
| **Híbrido** | Combinar los anteriores | Ambos | Lo usado en producción | Más complejo |

**La matriz usuario × ítem** es el objeto central: filas = estudiantes, columnas = cursos, celdas = rating. Casi toda está **vacía**; recomendar = **llenar los huecos**.

---
## 0. Preparación del entorno

In [ ]:
# Instala lo que falte en el MISMO Python que usa este kernel
import importlib.util, subprocess, sys

paquetes = {"tensorflow": "tensorflow", "sklearn": "scikit-learn", "pandas": "pandas", "matplotlib": "matplotlib"}
faltan = [pip for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
print("Python del kernel:", sys.executable)
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
    print("✓ Listo. Si algún import falla, reinicia el kernel.")
else:
    print("✓ Todo instalado.")

In [ ]:
import os, random, time, warnings
from collections import deque
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow import keras

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)

SEED = 42
def fijar_semilla(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
fijar_semilla()

print("TensorFlow", tf.__version__, "· GPU:", tf.config.list_physical_devices("GPU") or "no (CPU va bien)")

In [ ]:
# ---------------- CONFIGURACIÓN DE LA CLASE ----------------
RUTA_DATOS = Path("..") / "datasets"
K = 5                    # cuántos cursos recomendamos (top-K)
UMBRAL_RELEVANTE = 4     # rating >= 4 → "le gustó"
EPOCAS_TF = 40           # tope de épocas en A6 (EarlyStopping corta antes)

---
# 🅰️ Parte A · "Te puede interesar…": sistemas de recomendación

## A1. Los datos de la plataforma
| Archivo | Filas | Columnas clave |
|---|---|---|
| `cursos.csv` | 40 | `curso_id`, `titulo`, `area`, `nivel` (1–3), `etiquetas`, `descripcion` |
| `estudiantes.csv` | 600 | `estudiante_id`, `perfil`, `nivel_inicial` |
| `interacciones.csv` | ~7 900 | `estudiante_id`, `curso_id`, `rating` (1–5), `completado`, `fecha` |

In [ ]:
# ✍️ LIVE CODING — lee los 3 CSV de RUTA_DATOS (fecha como fecha) y muestra una muestra de cada uno
# A1.1 Cargamos las tres tablas


In [ ]:
# ✍️ LIVE CODING — calcula la densidad de la matriz y grafica ratings, cola larga y cursos por estudiante
# A1.2 ¿Qué tan vacía está la matriz usuario × curso? + cola larga


In [ ]:
# ✍️ LIVE CODING — une las tablas y haz un heatmap del rating promedio por perfil × área
# A1.3 La estructura escondida: rating promedio por perfil y área


> 💡 Cada perfil tiene gustos distintos. **El modelo no verá la columna `perfil`**: solo los ratings. Si un método colaborativo funciona, es porque *redescubre* estos grupos a partir del comportamiento.

### A1.4 Índices y separación entrenamiento / prueba
Para cada estudiante escondemos el **20 %** de sus calificaciones: el modelo aprende con el 80 % y lo evaluamos prediciendo lo escondido.

In [ ]:
# ✍️ LIVE CODING — crea u2i/c2i, columnas u e i, separa 20% por estudiante para test y arma la matriz R con train
# A1.4 Índices enteros (los modelos trabajan con 0..n-1) y split por estudiante


In [ ]:
# A1.5 Métricas y utilidades (esta celda viene lista)
def rmse(pred, datos=test):
    p = np.clip(pred[datos.u, datos.i], 1, 5)
    return float(np.sqrt(np.mean((p - datos.rating.to_numpy()) ** 2)))

relevantes_test = test[test.rating >= UMBRAL_RELEVANTE].groupby("u").i.apply(set)

def evaluar_ranking(scores, k=K):
    # Precision@k: de los k que recomendé, ¿cuántos le gustaron?  Recall@k: de los que le gustaron, ¿cuántos encontré?
    s = scores.astype(float).copy()
    s[visto] = -np.inf                         # nunca recomendamos lo que ya tomó
    top = np.argsort(-s, axis=1)[:, :k]
    prec, rec = [], []
    for u, rel in relevantes_test.items():
        hits = len(set(top[u]) & rel)
        prec.append(hits / k); rec.append(hits / len(rel))
    return {f"precision@{k}": np.mean(prec), f"recall@{k}": np.mean(rec), "cobertura": len(np.unique(top)) / n_items}

def recomendar(scores, estudiante_id, k=K):
    u = u2i[estudiante_id]
    s = scores[u].astype(float).copy(); s[visto[u]] = -np.inf
    top = np.argsort(-s)[:k]
    return cursos.iloc[top][["curso_id", "titulo", "area", "nivel"]].assign(score=s[top].round(3))

resultados = {}   # aquí guardamos cada método para compararlos en A7

## A2. Baseline: popularidad con promedio bayesiano
¿Recomendar el curso con mejor promedio? Un curso con **un solo 5** quedaría primero. El **promedio bayesiano** "encoge" hacia la media global a los cursos con pocas calificaciones:

$$\text{score}_i = \frac{C \cdot \bar{r} + n_i \cdot \bar{r}_i}{C + n_i}$$

donde $\bar r$ es la media global, $n_i$ las calificaciones del curso y $C$ cuánta "evidencia" exigimos (aquí, el promedio de calificaciones por curso). Así ordena IMDb su Top 250.

In [ ]:
# ✍️ LIVE CODING — calcula media global, count y mean por curso, el promedio bayesiano, y evalúa scores_pop
# A2.1 Popularidad: el mismo top para todos


## A3. Recomendador basado en contenido (TF-IDF)
1. Cada curso → vector **TF-IDF** de su título, área, etiquetas y descripción (lo mismo que en la Unidad 2).
2. **Similitud coseno** entre cursos: "parecidos por lo que dicen".
3. **Perfil del estudiante** = suma de los vectores de sus cursos, pesados por (rating − 3): lo que le gustó suma, lo que no le gustó resta.
4. Recomendamos los cursos más parecidos a su perfil.

In [ ]:
# ✍️ LIVE CODING — vectoriza título+área+etiquetas+descripción con TF-IDF, calcula coseno y una función similares()
# A3.1 TF-IDF de los cursos y cursos parecidos


In [ ]:
# ✍️ LIVE CODING — perfil = (R−3) sobre lo visto × TF-IDF; scores = coseno(perfil, cursos); evalúa
# A3.2 Perfil de cada estudiante y recomendación


## A4. Filtrado colaborativo item-item
"Los estudiantes que calificaron bien *Pandas* también calificaron bien *Visualización*." Ahora la similitud entre cursos sale del **comportamiento**, no del texto.

1. **Centramos** por estudiante (restamos su media): unos califican todo con 4–5, otros son exigentes.
2. Similitud coseno entre **columnas** de la matriz centrada (*adjusted cosine*).
3. Para predecir el rating de *u* en el curso *i*, promediamos sus ratings en los **k cursos más parecidos** a *i*:

$$\hat r_{ui} = \bar r_u + \frac{\sum_{j \in N_k(i)} s_{ij}\,(r_{uj} - \bar r_u)}{\sum_{j \in N_k(i)} |s_{ij}|}$$

In [ ]:
# ✍️ LIVE CODING — media por estudiante, matriz centrada Rc y coseno entre columnas (cursos)
# A4.1 Matriz centrada y similitud item-item


In [ ]:
# ✍️ LIVE CODING — deja solo los k=10 vecinos por curso y aplica la fórmula vectorizada (Rc @ S_k) / (visto @ |S_k|)
# A4.2 Predicción con los k vecinos más parecidos


## A5. Factorización de matrices con SVD (Scikit-Learn)
Idea clave del premio Netflix (2009): la matriz $R$ (600 × 40) se aproxima con dos matrices pequeñas:

$$R \approx U \cdot V^\top \qquad U: 600 \times k \quad V: 40 \times k$$

Cada estudiante y cada curso queda como un vector de **k factores latentes** ("cuánto de datos", "cuánto de web"…) que el modelo descubre solo. El rating es el producto punto: qué tanto coinciden.

In [ ]:
# ✍️ LIVE CODING — prueba TruncatedSVD con k en [2,4,8,16,32] sobre Rc, mide RMSE y guarda el de k=8
# A5.1 TruncatedSVD con distintos k


## A6. Recomendadores neuronales con TensorFlow
### A6.1 Factorización de matrices como red neuronal
- `Embedding(usuarios, d)` y `Embedding(cursos, d)` = las matrices $U$ y $V$ (las aprende el optimizador).
- Más un **sesgo** por usuario (exigente o generoso) y por curso (bueno o malo).
- Entrenamos **solo con las celdas observadas**. Como objetivo usamos la desviación respecto a la media global.

$$\hat r_{ui} = \bar r + b_u + b_i + \mathbf{p}_u \cdot \mathbf{q}_i$$

In [ ]:
# A6.0 Datos para Keras (esta celda viene lista)
def entradas(u, i):
    return {"usuario": np.asarray(u).reshape(-1, 1), "curso": np.asarray(i).reshape(-1, 1)}

tr = train.sample(frac=1, random_state=SEED)
corte = int(0.9 * len(tr))                         # 10 % de train para validación (EarlyStopping)
X_tr, y_tr = entradas(tr.u[:corte], tr.i[:corte]), (tr.rating[:corte].to_numpy() - media_global).reshape(-1, 1)
X_va, y_va = entradas(tr.u[corte:], tr.i[corte:]), (tr.rating[corte:].to_numpy() - media_global).reshape(-1, 1)

todos_u = np.repeat(np.arange(n_usuarios), n_items)
todos_i = np.tile(np.arange(n_items), n_usuarios)
def matriz_pred(modelo):
    p = modelo.predict(entradas(todos_u, todos_i), batch_size=4096, verbose=0)
    return p.reshape(n_usuarios, n_items) + media_global

def entrenar(modelo, lr=5e-3):
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss="mse", metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])
    parar = keras.callbacks.EarlyStopping(monitor="val_rmse", patience=4, restore_best_weights=True)
    t0 = time.time()
    h = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCAS_TF, batch_size=256, callbacks=[parar], verbose=0)
    print(f"{modelo.name}: {len(h.history['loss'])} épocas en {time.time() - t0:.0f}s · val_rmse {min(h.history['val_rmse']):.3f}")
    return h

In [ ]:
# ✍️ LIVE CODING — Inputs usuario/curso → Embeddings p_u, q_i y sesgos b_u, b_i → Dot + Add; entrena y evalúa
# A6.1 Modelo de factorización de matrices en Keras


### A6.2 Neural Collaborative Filtering (NCF)
El producto punto solo captura relaciones **lineales** entre factores. NCF (He et al., 2017) deja que una **red densa** aprenda la interacción. Su versión más usada, **NeuMF**, tiene dos ramas que se unen al final:
- **GMF:** producto elemento a elemento $\mathbf{p}_u \odot \mathbf{q}_i$ (la factorización, generalizada).
- **MLP:** los embeddings concatenados pasan por capas densas.

In [ ]:
# ✍️ LIVE CODING — rama GMF (Multiply) + rama MLP (Concatenate → Dense 32 → Dropout → Dense 16) + sesgos; entrena y evalúa
# A6.2 NeuMF: rama GMF + rama MLP + sesgos


In [ ]:
# ✍️ LIVE CODING — grafica rmse/val_rmse de ambos modelos y el PCA 2D de la capa emb_curso coloreado por área
# A6.3 Curvas de entrenamiento y lo que aprendieron los embeddings de los cursos


## A7. ¿Cuál ponemos en producción?

In [ ]:
# ✍️ LIVE CODING — DataFrame de resultados, resalta el mejor por métrica y grafica precision/recall/cobertura
# A7.1 Tabla comparativa


## A8. Arranque en frío y recomendador híbrido
**Arranque en frío (cold start):** un estudiante nuevo no tiene ratings → los colaborativos no saben qué decirle. Un curso nuevo nadie lo ha calificado → no aparece nunca.

**Híbrido ponderado:** mezclamos colaborativo y contenido según cuánto historial tiene cada estudiante:

$$\text{score} = \alpha_u \cdot \text{colaborativo} + (1-\alpha_u)\cdot\text{contenido}, \qquad \alpha_u = \frac{n_u}{n_u + 5}$$

In [ ]:
# ✍️ LIVE CODING — normaliza por fila, alpha = n_vistos/(n_vistos+5) y combina pred_mf con scores_contenido
# A8.1 Híbrido ponderado por historial


In [ ]:
# ✍️ LIVE CODING — TF-IDF del texto de intereses → coseno con los cursos + 20% popularidad, penaliza nivel 3
# A8.2 Un estudiante nuevo escribe qué quiere aprender (sin ningún rating)


In [ ]:
# ✍️ LIVE CODING — muestra qué le gustó a un estudiante y compara recomendar() con 4 métodos
# A8.3 Mismo estudiante, distintos recomendadores


---
# 🅱️ Parte B · Aprendizaje por refuerzo: decidir en secuencia

## B1. Teoría: agente, entorno y recompensa
En aprendizaje supervisado alguien nos da la respuesta correcta. En **aprendizaje por refuerzo (RL)** nadie la da: un **agente** prueba acciones, el **entorno** le responde con una **recompensa**, y el agente aprende una **política** que maximiza la recompensa **acumulada**.

| Concepto | Símbolo | En nuestro caso |
|---|---|---|
| Agente | — | El sistema de recomendación |
| Entorno | — | El estudiante / la plataforma |
| Estado | $s$ | Cursos que ya completó |
| Acción | $a$ | Qué curso recomendar ahora |
| Recompensa | $r$ | +clic, +curso completado, −abandono |
| Política | $\pi(a \mid s)$ | La regla para elegir la acción |
| Factor de descuento | $\gamma$ | Cuánto valen las recompensas futuras (0,95) |

**El dilema central: exploración vs explotación.** ¿Recomiendo lo que *sé* que funciona (explotar) o pruebo algo nuevo que quizá es mejor (explorar)?
Aplicaciones: AlphaGo, robots, conducción, trading, **RLHF** en ChatGPT y Claude (Unidad 2), y recomendadores de YouTube/TikTok/Spotify.

## B2. Multi-armed bandits: ¿qué curso va en el banner?
Una máquina tragamonedas con varios brazos (*arms*): cada uno paga con una probabilidad **desconocida**. Aquí cada brazo es un curso que podemos poner en el banner de la página de inicio; la recompensa es **1 si el visitante hace clic**.

| Estrategia | Cómo explora |
|---|---|
| **Greedy** | Nunca: siempre el mejor promedio hasta ahora |
| **ε-greedy** | Con probabilidad ε elige un brazo al azar |
| **UCB1** | Suma un bono de incertidumbre: $\bar x_a + \sqrt{2\ln t / n_a}$ ("optimismo ante la duda") |
| **Thompson sampling** | Muestrea de la distribución Beta de cada brazo y elige el mayor |

**Regret (arrepentimiento):** cuántos clics perdimos frente a haber sabido desde el inicio cuál era el mejor.

In [ ]:
# ✍️ LIVE CODING — CTR real de 6 cursos (0.5 × fracción de ratings ≥ 4) y una clase Banner con mostrar(a) → 0/1
# B2.1 El entorno: 6 cursos candidatos con una tasa de clic (CTR) real que el agente NO conoce


In [ ]:
# ✍️ LIVE CODING — clases Greedy (con eps opcional), UCB1 y Thompson con elegir(t) y actualizar(a, r)
# B2.2 Las políticas


In [ ]:
# ✍️ LIVE CODING — simular() acumula regret y % de brazo óptimo; corre las 4 políticas y grafica ambas curvas
# B2.3 Simulación: 30 corridas × 5 000 visitantes


## B3. Q-learning: la ruta de aprendizaje óptima
El bandit decide **una vez**. Ahora la decisión de hoy cambia el futuro: si un estudiante toma *Machine Learning* sin saber *Pandas*, probablemente falla y puede **abandonar**. Es un **proceso de decisión de Markov (MDP)**.

**El entorno `RutaAprendizajeEnv`:**
- **Estado:** qué cursos completó (8 bits → 256 estados).
- **Acción:** cuál de los 8 cursos tomar ahora.
- **Dinámica:** con los prerrequisitos aprueba el 90 % de las veces; sin ellos, el 20 %. Si reprueba, abandona con 15 % de probabilidad.
- **Recompensa:** −1 por cada paso (tiempo), +3 por curso aprobado, −2 por reprobar o repetir, −10 si abandona, **+20 al aprobar el proyecto final**.

**Q-learning** aprende $Q(s,a)$ = "qué tan bueno es tomar la acción $a$ en el estado $s$ y seguir bien después":

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,\big]$$

In [ ]:
# ✍️ LIVE CODING — estado = bitmask de cursos aprobados; step(a) aplica prerrequisitos, aprobar/reprobar/abandonar y recompensas
# B3.1 El entorno de la ruta de aprendizaje


In [ ]:
# ✍️ LIVE CODING — bucle de episodios con ε decreciente, acción ε-greedy, actualización de Bellman; grafica la media móvil
# B3.2 Q-learning tabular con ε decreciente


In [ ]:
# ✍️ LIVE CODING — política greedy sobre Q y una función que imprime un episodio paso a paso
# B3.3 ¿Qué aprendió? La ruta del agente, paso a paso


In [ ]:
# ✍️ LIVE CODING — compara aleatoria, 'más popular primero' y Q-learning con evaluar_politica() en 1000 episodios
# B3.4 Comparación con otras estrategias (1 000 estudiantes simulados cada una)


## B4. (Opcional) Deep Q-Network con TensorFlow
Con 256 estados una tabla basta. Con millones (un estudiante descrito por su historial, su perfil, su tiempo libre…) la tabla no cabe: usamos una **red neuronal** que recibe el estado y devuelve un $Q$ por acción (DeepMind, Atari 2015).

Dos trucos que la hacen estable:
- **Replay buffer:** guardamos transiciones y entrenamos con lotes aleatorios (rompe la correlación entre pasos seguidos).
- **Red objetivo:** una copia congelada calcula $r + \gamma \max Q(s',a')$; se sincroniza cada cierto número de pasos.

In [ ]:
# ✍️ LIVE CODING — red Q 8→64→64→8, red objetivo, replay buffer y paso_entrenamiento con Huber; entrena y evalúa
# B4.1 DQN


In [ ]:
# ✍️ LIVE CODING — grafica la media móvil de retornos de Q-learning y de DQN
# B4.2 Q-learning tabular vs DQN


---
## ✅ Cierre

| | Recomendación clásica (A) | Aprendizaje por refuerzo (B) |
|---|---|---|
| Pregunta | ¿Qué le gusta a este usuario? | ¿Qué hago ahora para maximizar el resultado **a largo plazo**? |
| Datos | Historial fijo (ratings) | Interacción: probar → recompensa |
| Aprende | A predecir / ordenar | Una **política** |
| Herramientas de hoy | TF-IDF, coseno, SVD, embeddings en TensorFlow | Bandits (ε-greedy, UCB, Thompson), Q-learning, DQN |

**Lo que aprendimos:** matriz usuario × ítem · popularidad bayesiana · contenido · colaborativo item-item · factorización (SVD y TensorFlow) · NCF · precision@k, recall@k, cobertura · arranque en frío e híbrido · exploración vs explotación · MDP y Q-learning · DQN.

### 🏋️ Reto / tarea
1. **Recomendador:** agrega al modelo de factorización el **perfil** y el **nivel** del estudiante como entradas extra (un modelo de "dos torres") y mide si mejora el arranque en frío.
2. **Bandit contextual:** repite B2 pero con un CTR distinto por perfil de estudiante; entrena un Thompson sampling **por perfil** y compáralo con uno global.
3. **RL:** añade al entorno un curso opcional *"Inglés técnico"* que sube la probabilidad de aprobar todo lo demás un 5 %. ¿El agente aprende a tomarlo primero? ¿Cambia con `gamma`?